In [3]:
#!/usr/bin/env python3
"""
Generate benchmark test data for Stars2Cells pipeline.

Structure:
  8 animals × 5 sessions × 5 seeds × 9 conditions

Tiers:
  Tier 0 (identity, 1 condition) – jitter only, single seed
    - T0_identity: no rotation, no translation, no dropout

  Tier A (sanity, 3 conditions) – easy ranges, fixed across sessions
    - A_rot:      rotation only
    - A_trans:    translation only
    - A_combined: rotation + translation

  Tier B (realistic, 4 conditions) – harder ranges, ONE factor varies
    - B_dropout:  dropout varies, rotation/drift at baseline
    - B_drift:    drift varies, dropout/rotation at baseline
    - B_rot:      rotation varies, dropout/drift at baseline
    - B_combined: all three vary

  Tier C (stress, 1 condition) – random walk, everything shifts each session
    - C_walk:     rotation, translation, dropout all change per session (±)

Neuron placement uses Poisson disk sampling (blue noise) for well-separated,
non-clustered layouts. Dropout models detection failure only — no new neurons
are spawned. FOV is 600×600. Translation is 2D. Neurons outside the FOV after
transforms are clipped (realistic detection loss).

Naming:
  {animal}_{session}__{condition}__seed{seed}.npy
"""

import numpy as np
from pathlib import Path
from dataclasses import dataclass
from typing import Optional


# ═══════════════════════════════════════════════════════════════════════════
# Spatial / transform helpers
# ═══════════════════════════════════════════════════════════════════════════

def poisson_disk_sample(n_neurons, fov_size=600, r_min=None, margin=10, seed=None):
    """
    Bridson's algorithm for Poisson disk sampling.

    Produces well-separated, non-clustered neuron layouts.  If *r_min* is
    None it is set to 55 % of the theoretical hex-packing maximum distance
    for the requested *n_neurons* inside the usable FOV area.
    """
    rng = np.random.RandomState(seed)

    lo = margin
    hi = fov_size - margin
    width = hi - lo

    if r_min is None:
        effective_area = width ** 2
        r_max = np.sqrt(2 * effective_area / (n_neurons * np.sqrt(3)))
        r_min = 0.55 * r_max

    cell_size = r_min / np.sqrt(2)
    grid_w = int(np.ceil(width / cell_size))
    grid = -np.ones((grid_w, grid_w), dtype=int)

    points = []
    active = []

    # --- first point ---
    p0 = np.array([rng.uniform(lo, hi), rng.uniform(lo, hi)])
    points.append(p0)
    active.append(0)
    gi = min(int((p0[0] - lo) / cell_size), grid_w - 1)
    gj = min(int((p0[1] - lo) / cell_size), grid_w - 1)
    grid[gi, gj] = 0

    k = 30  # candidates per active point

    while active and len(points) < n_neurons * 3:
        idx = rng.randint(len(active))
        point_idx = active[idx]
        center = points[point_idx]

        found = False
        for _ in range(k):
            angle = rng.uniform(0, 2 * np.pi)
            dist = rng.uniform(r_min, 2 * r_min)
            candidate = center + dist * np.array([np.cos(angle), np.sin(angle)])

            if candidate[0] < lo or candidate[0] >= hi or \
               candidate[1] < lo or candidate[1] >= hi:
                continue

            gi = min(int((candidate[0] - lo) / cell_size), grid_w - 1)
            gj = min(int((candidate[1] - lo) / cell_size), grid_w - 1)

            ok = True
            for di in range(-2, 3):
                for dj in range(-2, 3):
                    ni, nj = gi + di, gj + dj
                    if 0 <= ni < grid_w and 0 <= nj < grid_w and grid[ni, nj] >= 0:
                        if np.linalg.norm(candidate - points[grid[ni, nj]]) < r_min:
                            ok = False
                            break
                if not ok:
                    break

            if ok:
                new_idx = len(points)
                points.append(candidate)
                active.append(new_idx)
                grid[gi, gj] = new_idx
                found = True
                break

        if not found:
            active.pop(idx)

    points = np.array(points)
    if len(points) >= n_neurons:
        chosen = rng.choice(len(points), n_neurons, replace=False)
        points = points[chosen]
    else:
        # Fallback: rejection sampling with slightly relaxed r_min
        existing = list(points)
        attempts = 0
        while len(existing) < n_neurons and attempts < 200_000:
            p = np.array([rng.uniform(lo, hi), rng.uniform(lo, hi)])
            dists = np.linalg.norm(np.array(existing) - p, axis=1)
            if dists.min() >= r_min * 0.8:
                existing.append(p)
            attempts += 1
        points = np.array(existing[:n_neurons])

    return points[:, 0], points[:, 1]


def generate_base_jitter(n_neurons, jitter_std=0.5, seed=None):
    """
    Per-neuron centroid estimation bias — stable across sessions.

    Models the systematic offset CNMF introduces when estimating each
    cell's centroid relative to its true centre.
    """
    rng = np.random.RandomState(seed)
    dx = rng.normal(0, jitter_std, n_neurons)
    dy = rng.normal(0, jitter_std, n_neurons)
    return dx, dy


def add_session_jitter(base_jitter_x, base_jitter_y,
                       session_perturbation_std=0.2, seed=None):
    """
    Apply base jitter + a small per-session perturbation on top.

    Returns the total (x, y) jitter offsets for this session.
    """
    rng = np.random.RandomState(seed)
    n = len(base_jitter_x)
    dx = base_jitter_x + rng.normal(0, session_perturbation_std, n)
    dy = base_jitter_y + rng.normal(0, session_perturbation_std, n)
    return dx, dy


def simulate_dropout(x, y, roi_ids, dropout_rate=0.05, seed=None):
    """
    Simulate detection failure — neurons are simply lost, no replacements.

    Models CNMF missing a cell due to low SNR, inactivity, or motion
    artefact in a given session.
    """
    if dropout_rate <= 0:
        return x.copy(), y.copy(), roi_ids.copy()
    rng = np.random.RandomState(seed)
    n = len(x)
    n_dropout = int(n * dropout_rate)
    if n_dropout == 0:
        return x.copy(), y.copy(), roi_ids.copy()
    keep_mask = np.ones(n, dtype=bool)
    keep_mask[rng.choice(n, n_dropout, replace=False)] = False
    return x[keep_mask], y[keep_mask], roi_ids[keep_mask]


def rotate_neurons(x, y, angle_deg, center=None):
    if center is None:
        center = (np.mean(x), np.mean(y))
    rad = np.deg2rad(angle_deg)
    xc, yc = x - center[0], y - center[1]
    cos_a, sin_a = np.cos(rad), np.sin(rad)
    return xc * cos_a - yc * sin_a + center[0], \
           xc * sin_a + yc * cos_a + center[1]


def translate_neurons(x, y, dx, dy):
    return x + dx, y + dy


def clip_to_fov(x, y, roi_ids, fov_size=600):
    """Remove neurons that landed outside the sensor after transforms."""
    mask = (x >= 0) & (x < fov_size) & (y >= 0) & (y < fov_size)
    return x[mask], y[mask], roi_ids[mask]


def permute_ids(roi_ids):
    return roi_ids[np.random.permutation(len(roi_ids))]


# ═══════════════════════════════════════════════════════════════════════════
# Tier parameter dataclasses
# ═══════════════════════════════════════════════════════════════════════════

@dataclass
class Tier0Params:
    """
    Tier 0: Identity — jitter only, single seed.

    Sanity floor: S2C should score ~100 % here.
    """
    enabled: bool = True


@dataclass
class TierAParams:
    """
    Tier A: Sanity — easy parameter ranges, fixed across sessions.

    3 conditions: rotation only, translation only, combined.
    """
    rot_range: tuple = (0.5, 3.0)          # deg per session step
    trans_range: tuple = (1.0, 5.0)        # px per session step (magnitude)
    dropout_range: tuple = (0.01, 0.05)    # detection-failure fraction
    enabled: bool = True


@dataclass
class TierBParams:
    """
    Tier B: Realistic — harder ranges, ONE factor isolated per condition.

    4 conditions: dropout only, drift only, rotation only, combined.
    Non-varying factors use baseline values.
    """
    rot_range: tuple = (2.0, 10.0)         # deg per session step
    trans_range: tuple = (5.0, 25.0)       # px per session step (magnitude)
    dropout_range: tuple = (0.05, 0.30)    # detection-failure fraction

    baseline_rot: float = 2.0
    baseline_trans: float = 5.0
    baseline_dropout: float = 0.05
    enabled: bool = True


@dataclass
class TierCParams:
    """
    Tier C: Stress — random walk, everything shifts per session.

    1 condition: rotation, translation, and dropout all change each
    session with random magnitude and direction (±).
    """
    rot_step_range: tuple = (2.0, 8.0)
    trans_step_range: tuple = (3.0, 15.0)
    dropout_range: tuple = (0.10, 0.35)
    enabled: bool = True


# ═══════════════════════════════════════════════════════════════════════════
# Core generator
# ═══════════════════════════════════════════════════════════════════════════

def generate_benchmark(
    output_dir: str = "Stars2Cells_Sample_Data",
    n_animals: int = 8,
    n_neurons: int = 120,
    n_seeds: int = 5,
    sessions_per_animal: int = 5,
    base_jitter_std: float = 0.5,
    session_perturbation_std: float = 0.2,
    tier_0_params: Optional[Tier0Params] = None,
    tier_a_params: Optional[TierAParams] = None,
    tier_b_params: Optional[TierBParams] = None,
    tier_c_params: Optional[TierCParams] = None,
    fov_size: int = 600,
    r_min: Optional[float] = None,
    verbose: bool = True,
) -> dict:
    """
    Generate the full benchmark dataset.

    Returns
    -------
    dict with keys: 'file_count', 'n_conditions', 'output_dir', 'manifest'
    """
    tier_0 = tier_0_params or Tier0Params()
    tier_a = tier_a_params or TierAParams()
    tier_b = tier_b_params or TierBParams()
    tier_c = tier_c_params or TierCParams()

    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True, parents=True)

    n_sess = sessions_per_animal
    center = (fov_size / 2, fov_size / 2)
    file_count = 0
    manifest = []
    ground_truth = {}

    # ── Build condition list ──
    # (name, tier_key, which_factors_vary, n_seeds_override)
    conditions = []
    if tier_0.enabled:
        conditions.append(('T0_identity', '0', set(), 1))  # single seed
    if tier_a.enabled:
        conditions.append(('A_rot',      'A', {'rot'},          n_seeds))
        conditions.append(('A_trans',    'A', {'trans'},         n_seeds))
        conditions.append(('A_combined', 'A', {'rot', 'trans'}, n_seeds))
    if tier_b.enabled:
        conditions.append(('B_dropout',  'B', {'dropout'},                  n_seeds))
        conditions.append(('B_drift',    'B', {'trans'},                    n_seeds))
        conditions.append(('B_rot',      'B', {'rot'},                      n_seeds))
        conditions.append(('B_combined', 'B', {'rot', 'trans', 'dropout'}, n_seeds))
    if tier_c.enabled:
        conditions.append(('C_walk',     'C', {'rot', 'trans', 'dropout'}, n_seeds))

    if verbose:
        tier_counts = {}
        for _, t, _, _ in conditions:
            tier_counts[t] = tier_counts.get(t, 0) + 1
        print("Stars2Cells Benchmark Data Generator  (v2)")
        print("═" * 55)
        print(f"  Animals              : {n_animals}")
        print(f"  Neurons/animal       : {n_neurons}")
        print(f"  Seeds/condition      : {n_seeds}")
        print(f"  Sessions/animal      : {n_sess}")
        print(f"  FOV                  : {fov_size}×{fov_size}")
        print(f"  r_min (Poisson disk) : {r_min or 'auto'}")
        print(f"  Base jitter std      : {base_jitter_std}")
        print(f"  Session perturb std  : {session_perturbation_std}")
        print(f"  Conditions           : {len(conditions)} total")
        for t in sorted(tier_counts):
            label = '0 (identity)' if t == '0' else t
            print(f"    Tier {label}: {tier_counts[t]}")
        print("═" * 55)

    for animal_idx in range(n_animals):
        animal_id = f"{n_neurons + animal_idx + 1}"
        animal_seed = animal_idx * 10_000

        # ── Poisson disk placement ──
        x_base, y_base = poisson_disk_sample(
            n_neurons, fov_size=fov_size, r_min=r_min, seed=animal_seed,
        )

        # ── Per-neuron base jitter (stable across sessions) ──
        base_jx, base_jy = generate_base_jitter(
            len(x_base), jitter_std=base_jitter_std, seed=animal_seed + 1,
        )

        roi_ids_base = np.arange(len(x_base))

        for cond_name, tier_key, varies, cond_n_seeds in conditions:

            for seed_idx in range(cond_n_seeds):
                param_seed = animal_seed + hash(cond_name) % 10_000 + seed_idx * 1000
                rng = np.random.RandomState(param_seed)

                # ── Draw parameters based on tier ──

                if tier_key == '0':
                    # Identity: jitter only
                    manifest.append(dict(
                        animal=animal_id, condition=cond_name, seed=seed_idx,
                        rot_step=0.0, trans_mag=0.0, trans_angle_deg=0.0,
                        dropout_rate=0.0,
                    ))

                elif tier_key == 'A':
                    rot_step = rng.uniform(*tier_a.rot_range) if 'rot' in varies else 0.0
                    trans_mag = rng.uniform(*tier_a.trans_range) if 'trans' in varies else 0.0
                    trans_angle = rng.uniform(0, 2 * np.pi)  # fixed dir for this seed
                    dropout_rate = rng.uniform(*tier_a.dropout_range)

                    manifest.append(dict(
                        animal=animal_id, condition=cond_name, seed=seed_idx,
                        rot_step=rot_step, trans_mag=trans_mag,
                        trans_angle_deg=np.rad2deg(trans_angle),
                        dropout_rate=dropout_rate,
                    ))

                elif tier_key == 'B':
                    rot_step = rng.uniform(*tier_b.rot_range) if 'rot' in varies \
                               else tier_b.baseline_rot
                    trans_mag = rng.uniform(*tier_b.trans_range) if 'trans' in varies \
                                else tier_b.baseline_trans
                    trans_angle = rng.uniform(0, 2 * np.pi)
                    dropout_rate = rng.uniform(*tier_b.dropout_range) if 'dropout' in varies \
                                   else tier_b.baseline_dropout

                    manifest.append(dict(
                        animal=animal_id, condition=cond_name, seed=seed_idx,
                        rot_step=rot_step, trans_mag=trans_mag,
                        trans_angle_deg=np.rad2deg(trans_angle),
                        dropout_rate=dropout_rate,
                    ))

                elif tier_key == 'C':
                    # Random walk: per-session steps with random direction
                    rot_steps = rng.uniform(*tier_c.rot_step_range, n_sess) \
                                * rng.choice([-1, 1], n_sess)
                    trans_mags = rng.uniform(*tier_c.trans_step_range, n_sess)
                    trans_angles = rng.uniform(0, 2 * np.pi, n_sess)
                    dropout_per_sess = rng.uniform(*tier_c.dropout_range, n_sess)

                    rot_cumulative = np.concatenate([[0.0], np.cumsum(rot_steps)])
                    # 2D cumulative translation
                    dx_steps = trans_mags * np.cos(trans_angles)
                    dy_steps = trans_mags * np.sin(trans_angles)
                    dx_cumulative = np.concatenate([[0.0], np.cumsum(dx_steps)])
                    dy_cumulative = np.concatenate([[0.0], np.cumsum(dy_steps)])

                    manifest.append(dict(
                        animal=animal_id, condition=cond_name, seed=seed_idx,
                        rot_walk=rot_cumulative[1:].tolist(),
                        dx_walk=dx_cumulative[1:].tolist(),
                        dy_walk=dy_cumulative[1:].tolist(),
                        dropout_per_sess=dropout_per_sess.tolist(),
                    ))

                # ── Generate sessions ──

                for sess in range(1, n_sess + 1):

                    if tier_key == '0':
                        rotation_deg = 0.0
                        dx, dy = 0.0, 0.0
                        dropout_r = 0.0
                    elif tier_key == 'C':
                        rotation_deg = rot_cumulative[sess]
                        dx = dx_cumulative[sess]
                        dy = dy_cumulative[sess]
                        dropout_r = dropout_per_sess[sess - 1]
                    else:
                        rotation_deg = rot_step * (sess - 1)
                        mag = trans_mag * (sess - 1)
                        dx = mag * np.cos(trans_angle)
                        dy = mag * np.sin(trans_angle)
                        dropout_r = dropout_rate

                    # ── Build session ──
                    x, y = x_base.copy(), y_base.copy()
                    roi_ids = roi_ids_base.copy()

                    # Rotation
                    if rotation_deg != 0:
                        x, y = rotate_neurons(x, y, rotation_deg, center)

                    # 2D translation
                    if dx != 0 or dy != 0:
                        x, y = translate_neurons(x, y, dx, dy)

                    # Clip to FOV (neurons outside sensor are lost)
                    x, y, roi_ids = clip_to_fov(x, y, roi_ids, fov_size)

                    # Dropout (detection failure, session ≥ 2 only)
                    if sess >= 2 and dropout_r > 0:
                        x, y, roi_ids = simulate_dropout(
                            x, y, roi_ids, dropout_r,
                            seed=param_seed + sess * 7,
                        )

                    # Per-neuron jitter (base bias + session perturbation)
                    # Only apply to neurons that survived clipping/dropout
                    surviving_base_indices = roi_ids  # these ARE the base indices
                    sess_jx, sess_jy = add_session_jitter(
                        base_jx[surviving_base_indices],
                        base_jy[surviving_base_indices],
                        session_perturbation_std=session_perturbation_std,
                        seed=param_seed + sess * 13,
                    )
                    x = x + sess_jx
                    y = y + sess_jy

                    # Ground truth: capture base identity BEFORE permutation
                    ground_truth_base_ids = roi_ids.copy()

                    roi_ids = permute_ids(roi_ids)

                    # ── Save ──
                    fname = f"{animal_id}_{sess}__{cond_name}__seed{seed_idx}.npy"
                    data = {
                        'centroids_x': np.array(x),
                        'centroids_y': np.array(y),
                        'roi_ids': np.array(roi_ids),
                        'subject_id': str(animal_id),
                        'session_id': str(sess),
                    }
                    np.save(output_dir / fname, data, allow_pickle=True)
                    file_count += 1

                    ground_truth[fname] = {
                        'ground_truth_base_ids': np.array(ground_truth_base_ids),
                        'permuted_roi_ids': np.array(roi_ids),
                    }

        if verbose:
            print(f"  Animal {animal_id} done ({file_count} files so far)")

    if verbose:
        print(f"\n  Done — {file_count} files in {output_dir.absolute()}")

    # ── Save ground truth ──
    gt_path = output_dir / "ground_truth.npy"
    np.save(gt_path, ground_truth, allow_pickle=True)
    if verbose:
        print(f"  Ground truth saved → {gt_path}")

    return {
        'file_count': file_count,
        'n_conditions': len(conditions),
        'output_dir': str(output_dir.absolute()),
        'manifest': manifest,
    }


# ═══════════════════════════════════════════════════════════════════════════
# CLI entry point
# ═══════════════════════════════════════════════════════════════════════════
def main():
    base = Path(r"C:\Users\ariAccount\Desktop")

    # # ── 100n and 250n ─────────────────────────────────────────────────────
    # for n in [100, 250]:
    #     out = base / f"Stars2Cells_Benchmark_{n}n"
    #     if out.exists():
    #         print(f"\n  SKIP {n}n — {out} already exists\n")
    #     else:
    #         print(f"\n{'█' * 60}")
    #         print(f"  GENERATING: {n} neurons")
    #         print(f"{'█' * 60}\n")
    #         result = generate_benchmark(
    #             output_dir=str(out),
    #             n_neurons=n,
    #             n_animals=8,
    #             n_seeds=5,
    #             sessions_per_animal=5,
    #             fov_size=600,
    #             base_jitter_std=0.5,
    #             session_perturbation_std=0.2,
    #             tier_0_params=Tier0Params(enabled=True),
    #             tier_a_params=TierAParams(enabled=True),
    #             tier_b_params=TierBParams(enabled=True),
    #             tier_c_params=TierCParams(enabled=True),
    #         )
    #         print(f"\n  ✓ {n}n — {result['file_count']} files → {result['output_dir']}\n")

    # # ── 500n ──────────────────────────────────────────────────────────────
    # n = 500
    # out = base / f"Stars2Cells_Benchmark_{n}n"
    # if out.exists():
    #     print(f"\n  SKIP {n}n — {out} already exists\n")
    # else:
    #     print(f"\n{'█' * 60}")
    #     print(f"  GENERATING: {n} neurons")
    #     print(f"{'█' * 60}\n")
    #     result = generate_benchmark(
    #         output_dir=str(out),
    #         n_neurons=n,
    #         n_animals=8,
    #         n_seeds=5,
    #         sessions_per_animal=5,
    #         fov_size=600,
    #         base_jitter_std=0.5,
    #         session_perturbation_std=0.2,
    #         tier_0_params=Tier0Params(enabled=True),
    #         tier_a_params=TierAParams(enabled=True),
    #         tier_b_params=TierBParams(enabled=True),
    #         tier_c_params=TierCParams(enabled=True),
    #     )
    #     print(f"\n  ✓ {n}n — {result['file_count']} files → {result['output_dir']}\n")

    # # ── 1000n: Tier 0 + Tier A only ──────────────────────────────────────
    # n = 1000
    # out = base / f"Stars2Cells_Benchmark_{n}n"
    # if out.exists():
    #     print(f"\n  SKIP {n}n — {out} already exists\n")
    # else:
    #     print(f"\n{'█' * 60}")
    #     print(f"  GENERATING: {n} neurons  (Tier 0 + A only)")
    #     print(f"{'█' * 60}\n")
    #     result = generate_benchmark(
    #         output_dir=str(out),
    #         n_neurons=n,
    #         n_animals=8,
    #         n_seeds=5,
    #         sessions_per_animal=5,
    #         fov_size=600,
    #         base_jitter_std=0.5,
    #         session_perturbation_std=0.2,
    #         tier_0_params=Tier0Params(enabled=True),
    #         tier_a_params=TierAParams(enabled=True),
    #         tier_b_params=TierBParams(enabled=False),
    #         tier_c_params=TierCParams(enabled=False),
    #     )
    #     print(f"\n  ✓ {n}n — {result['file_count']} files → {result['output_dir']}\n")
    
    # # ── 1000n: Tier B only ───────────────────────────────────────────────
    # out = Path(r"C:\Users\ariAccount\Desktop\Stars2CellsPaper\Stars2Cells_Benchmark_1000n_Tier_B")
    # if out.exists():
    #     print(f"\n  SKIP 1000n Tier B — {out} already exists\n")
    # else:
    #     print(f"\n{'█' * 60}")
    #     print(f"  GENERATING: 1000 neurons  (Tier B only)")
    #     print(f"{'█' * 60}\n")
    #     result = generate_benchmark(
    #         output_dir=str(out),
    #         n_neurons=1000,
    #         n_animals=8,
    #         n_seeds=5,
    #         sessions_per_animal=5,
    #         fov_size=600,
    #         base_jitter_std=0.5,
    #         session_perturbation_std=0.2,
    #         tier_0_params=Tier0Params(enabled=False),
    #         tier_a_params=TierAParams(enabled=False),
    #         tier_b_params=TierBParams(enabled=True),
    #         tier_c_params=TierCParams(enabled=False),
    #     )
    #     print(f"\n  ✓ 1000n Tier B — {result['file_count']} files → {result['output_dir']}\n")

    # ── 1000n: Tier C only ───────────────────────────────────────────────
    out = Path(r"C:\Users\ariAccount\Desktop\Stars2CellsPaper\Stars2Cells_Benchmark_1000n_Tier_C")
    if out.exists():
        print(f"\n  SKIP 1000n Tier C — {out} already exists\n")
    else:
        print(f"\n{'█' * 60}")
        print(f"  GENERATING: 1000 neurons  (Tier C only)")
        print(f"{'█' * 60}\n")
        result = generate_benchmark(
            output_dir=str(out),
            n_neurons=1000,
            n_animals=8,
            n_seeds=5,
            sessions_per_animal=5,
            fov_size=600,
            base_jitter_std=0.5,
            session_perturbation_std=0.2,
            tier_0_params=Tier0Params(enabled=False),
            tier_a_params=TierAParams(enabled=False),
            tier_b_params=TierBParams(enabled=False),
            tier_c_params=TierCParams(enabled=True),
        )
        print(f"\n  ✓ 1000n Tier C — {result['file_count']} files → {result['output_dir']}\n")


if __name__ == "__main__":
    main()


████████████████████████████████████████████████████████████
  GENERATING: 1000 neurons  (Tier C only)
████████████████████████████████████████████████████████████

Stars2Cells Benchmark Data Generator  (v2)
═══════════════════════════════════════════════════════
  Animals              : 8
  Neurons/animal       : 1000
  Seeds/condition      : 5
  Sessions/animal      : 5
  FOV                  : 600×600
  r_min (Poisson disk) : auto
  Base jitter std      : 0.5
  Session perturb std  : 0.2
  Conditions           : 1 total
    Tier C: 1
═══════════════════════════════════════════════════════
  Animal 1001 done (25 files so far)
  Animal 1002 done (50 files so far)
  Animal 1003 done (75 files so far)
  Animal 1004 done (100 files so far)
  Animal 1005 done (125 files so far)
  Animal 1006 done (150 files so far)
  Animal 1007 done (175 files so far)
  Animal 1008 done (200 files so far)

  Done — 200 files in C:\Users\ariAccount\Desktop\Stars2CellsPaper\Stars2Cells_Benchmark_1000n_Tie